# 🚀 Notebook 04 – XGBoost Final Model & Hyperparameter Tuning
**Project:** Student Performance Prediction Using ML
**Intern:** SEERAPU SRAVANI | 24U45A4219 | CSE-AIML, JNTUGV

This notebook tunes XGBoost, saves the final model, and generates all evaluation plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      RandomizedSearchCV)
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/student_data.csv')
df = df.drop(columns=['StudentID','FinalMarks','FinalGrade'])
le = LabelEncoder()
for col in ['Gender','SocioeconomicStatus','ParentEducation']:
    df[col] = le.fit_transform(df[col])
X = df.drop(columns=['Result'])
y = le.fit_transform(df['Result'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print('Data loaded. Features:', X.columns.tolist())

## 2. Hyperparameter Tuning – RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators'    : [100, 150, 200, 300],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.15],
    'max_depth'       : [3, 5, 6, 7, 8],
    'subsample'       : [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma'           : [0, 0.1, 0.2, 0.3],
}

xgb_base = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
rs = RandomizedSearchCV(xgb_base, param_dist, n_iter=40, cv=5,
                         scoring='accuracy', n_jobs=-1, random_state=42)
rs.fit(X_train, y_train)
print('Best params:', rs.best_params_)
print(f'Best CV accuracy: {rs.best_score_*100:.2f}%')

## 3. Final Model Evaluation

In [ ]:
best_model = rs.best_estimator_
y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
cv  = cross_val_score(best_model, X_scaled, y, cv=10, scoring='accuracy')

print('='*50)
print(f'  Final XGBoost Model Results')
print('='*50)
print(f'  Test Accuracy : {acc*100:.2f}%')
print(f'  ROC-AUC       : {auc:.4f}')
print(f'  10-Fold CV    : {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%')
print('='*50)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Fail','Pass']))

## 4. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Final XGBoost Model – Evaluation', fontsize=14, fontweight='bold')

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Fail','Pass'], yticklabels=['Fail','Pass'],
            cbar=False, linewidths=0.5, linecolor='white')
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#2E74B5', lw=2.5, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#2E74B5')
axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1.02])
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(loc='lower right')

plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance

In [ ]:
fi = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
fi.plot(kind='barh', ax=ax, color='#2E74B5', edgecolor='white')
ax.set_title('Feature Importance – XGBoost', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
for i, (val, name) in enumerate(zip(fi.values, fi.index)):
    ax.text(val+0.002, i, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 3 features:', fi.sort_values(ascending=False).head(3).index.tolist())

## 6. Save Final Model

In [ ]:
joblib.dump(best_model,        '../models/model.pkl')
joblib.dump(scaler,            '../models/scaler.pkl')
joblib.dump(list(X.columns),   '../models/feature_names.pkl')
print('Model saved to models/model.pkl')
print('Scaler saved to models/scaler.pkl')
print('Feature names saved to models/feature_names.pkl')
print(f'\nFinal model ready for deployment!')
print(f'Accuracy: {acc*100:.2f}% | AUC: {auc:.4f}')

## ✅ Final Model Complete
- XGBoost tuned with RandomizedSearchCV (40 iterations, 5-fold CV)
- Saved as model.pkl + scaler.pkl for Streamlit app deployment
- Run `streamlit run app/streamlit_app.py` to launch the prediction app